# Model Configuration & Providers

This notebook accompanies the [Agent Foundry](https://agent-foundry-pi.vercel.app) OpenAI Agents SDK roadmap.

You will learn how to configure model parameters with `ModelSettings`, switch API types, assign per-agent and per-run models, use `MultiProvider` for prefix routing, and integrate non-OpenAI providers through LiteLLM.

## 1. Install Dependencies

In [ ]:
!pip install -q openai-agents

## 2. Set Up Your API Key

In [ ]:
import os
from getpass import getpass

os.environ["OPENAI_API_KEY"] = getpass("Enter your OpenAI API key: ")

## 3. ModelSettings

Configure inference parameters like temperature, top_p, tool_choice, and parallel_tool_calls.

In [ ]:
from agents import Agent, Runner, ModelSettings

creative_agent = Agent(
    name="Creative Writer",
    instructions="You write creative fiction with vivid imagery.",
    model_settings=ModelSettings(
        temperature=0.9,
        top_p=0.95,
        tool_choice="auto",
        parallel_tool_calls=True,
    ),
)

result = await Runner.run(creative_agent, "Write a haiku about the ocean.")
print(result.final_output)

## 4. Setting the Default API Type

Switch all agents from the Responses API to the Chat Completions API.

In [ ]:
from agents import set_default_openai_api

set_default_openai_api("chat_completions")

agent = Agent(
    name="Chat Agent",
    instructions="You are a helpful assistant using the Chat Completions API.",
)

result = await Runner.run(agent, "What is the capital of France?")
print(result.final_output)

## 5. Per-Agent Model with OpenAIChatCompletionsModel

Assign a specific model and client to individual agents.

In [ ]:
from agents.models.openai_chatcompletions import OpenAIChatCompletionsModel
from openai import AsyncOpenAI

client = AsyncOpenAI()

fast_agent = Agent(
    name="Fast Responder",
    instructions="You give quick, concise answers.",
    model=OpenAIChatCompletionsModel(
        model="gpt-4o-mini",
        openai_client=client,
    ),
)

smart_agent = Agent(
    name="Deep Thinker",
    instructions="You provide thorough, well-reasoned analysis.",
    model=OpenAIChatCompletionsModel(
        model="gpt-4o",
        openai_client=client,
    ),
)

fast_result = await Runner.run(fast_agent, "What is 2+2?")
print(f"Fast: {fast_result.final_output}")

smart_result = await Runner.run(smart_agent, "Explain the implications of Gödel's incompleteness theorems.")
print(f"Smart: {smart_result.final_output}")

## 6. Per-Run Override with RunConfig

Override the model for a specific run without changing the agent definition.

In [ ]:
from agents import RunConfig

base_agent = Agent(
    name="Flexible Agent",
    instructions="You are a helpful assistant.",
)

result_mini = await Runner.run(
    base_agent,
    "What is the capital of France?",
    run_config=RunConfig(model="gpt-4o-mini"),
)
print(f"gpt-4o-mini: {result_mini.final_output}")

result_full = await Runner.run(
    base_agent,
    "Compare and contrast existentialism and absurdism.",
    run_config=RunConfig(model="gpt-4o"),
)
print(f"gpt-4o: {result_full.final_output}")

## 7. MultiProvider for Prefix Routing

Route model requests to different backends based on a prefix in the model name.

In [ ]:
from agents.models.multi_provider import MultiProvider
from agents.models.openai_provider import OpenAIProvider

multi = MultiProvider(
    providers=[
        OpenAIProvider(prefix="openai/"),
    ],
)

routed_agent = Agent(
    name="Router Agent",
    instructions="You are a helpful assistant.",
)

result = await Runner.run(
    routed_agent,
    "Hello!",
    run_config=RunConfig(
        model="openai/gpt-4o-mini",
        model_provider=multi,
    ),
)
print(result.final_output)

## 8. LiteLLM Adapter for Non-OpenAI Providers

Connect to Anthropic, Google, Mistral, and other providers through LiteLLM. Install the extension first.

In [ ]:
!pip install -q "openai-agents[litellm]"

In [ ]:
from agents.extensions.litellm_provider import LitellmModel

litellm_agent = Agent(
    name="LiteLLM Agent",
    instructions="You are a helpful assistant.",
    model=LitellmModel(model="anthropic/claude-sonnet-4-20250514"),
)

# Requires ANTHROPIC_API_KEY in environment
# result = await Runner.run(litellm_agent, "Explain quantum computing simply.")
# print(result.final_output)

print("LiteLLM agent configured. Set ANTHROPIC_API_KEY to run.")

## 9. Retry Policies

Configure automatic retries for transient API errors.

In [ ]:
config = RunConfig(
    max_retries=3,
    retry_delay=1.0,
)

result = await Runner.run(base_agent, "Tell me a joke.", run_config=config)
print(result.final_output)

## Key Takeaways

- Use `ModelSettings` to tune temperature, top_p, tool_choice, and parallel_tool_calls per agent
- Call `set_default_openai_api("chat_completions")` to switch all agents to the Chat Completions API
- Use `OpenAIChatCompletionsModel` to assign a specific model and client to an individual agent
- Override models per run with `RunConfig(model=)` for dynamic model selection
- Use `MultiProvider` to route requests to different backends based on model name prefixes
- Integrate non-OpenAI providers through the LiteLLM adapter